In [1]:
import kagglehub
kagglehub.login()

In [2]:
barnobarno_nemotron_high_reasoning_pass_1_and_2_and_3_and_4_path = kagglehub.dataset_download('barnobarno/nemotron-high-reasoning-pass-1-and-2-and-3-and-4')
#barnobarno_gpt_oss_120b_bnb_4bit_transformers_unsloth_1_path = kagglehub.model_download('barnobarno/gpt-oss-120b-bnb-4bit/Transformers/unsloth/1')

print('Data source import complete.')


100%|██████████| 5.53G/5.53G [04:21<00:00, 22.7MB/s]  

Extracting files...


Data source import complete.


In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024*4
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [7]:
import os 
os.listdir(barnobarno_nemotron_high_reasoning_pass_1_and_2_and_3_and_4_path)
data_path = os.path.join(barnobarno_nemotron_high_reasoning_pass_1_and_2_and_3_and_4_path, 'High_low_pass.jsonl')

In [8]:
class CONFIG:
    TRAIN_SIZE = 10_000
    BATCH_SIZE = 16 
    EPOCHS = 1 
    LEARNING_RATE = 2e-4
    MAX_SEQ_LENGTH = 4096*2
    MODEL_PATH: str | None = None
    KAGGLE=True 
    MASK_THINK = True
    KEEP_UNIQUE = False
    SEED = 42
    REASONING_EFFORT = "high"
    SAVE_STEPS = 100
    SAVE_TOTAL_LIMIT = 2
    RESUME_FROM_CHECKPOINT = False
    CHECKPOINT_PATH = None
    MERGE_INPUT_FILES = False
    DEFAULT_INPUT_FILE = data_path
    MERGE_INPUT_FILES_LIST = [
        "/kaggle/input/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_low_pass.jsonl",
        "/kaggle/input/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_medium_pass.jsonl",
    ]
    

cfg = CONFIG()
cfg.MODEL_PATH = "/kaggle/input/gpt-oss-20b-bnb-4bit/transformers/unsloth/1" if cfg.KAGGLE else "unsloth/gpt-oss-20b"

In [ ]:
import polars as pl

files = cfg.MERGE_INPUT_FILES_LIST if cfg.MERGE_INPUT_FILES else [cfg.DEFAULT_INPUT_FILE]
print(f"MERGE_INPUT_FILES={cfg.MERGE_INPUT_FILES} | Files: {files}")

def load_all_columns_lazy(file_path):
    # 1. infer_schema_length=None forces it to scan ALL rows to find sparse columns like 'tools'
    lazy_df = pl.scan_ndjson(file_path, infer_schema_length=None, ignore_errors=True)
    
    # 2. "Reverse DropNA" Logic
    # We perform the filter HERE (lazily) instead of after loading.
    # This gives you the same result as "filtering later" but saves the RAM 
    # that would have been wasted loading the 'tools' rows.
    schema_keys = lazy_df.collect_schema().names()
    
    if "tools" in schema_keys:
        lazy_df = lazy_df.filter(pl.col("tools").is_null())
    
    # 3. REMOVED .select() -> Now returns ALL columns
    return lazy_df

# Setup scans
queries = [load_all_columns_lazy(file_path) for file_path in files]

# Collect
print("Scanning and filtering (keeping all columns)...")

# CRITICAL: how="diagonal" ensures that if 'metadata' or 'url' is missing 
# in one file but present in the other, it doesn't crash.
df = pl.concat(queries, how="diagonal").collect()

print(f"Loaded {len(df)} rows with columns: {df.columns}")
df.head()

In [ ]:
df.head(1)

In [12]:
import polars as pl
import random
import numpy as np
import torch

# --- CONFIGURATION ---
KEEP_INT_ONLY = True
CHANGE_SYSTEM_PROMPT = False 

# Used only when CHANGE_SYSTEM_PROMPT = True
NEW_SYSTEM_PROMPT = (
    "You are an expert Math Olympiad solver. Your goal is to solve complex "
    "mathematical problems with rigorous, step-by-step reasoning.\n"
    "Knowledge cutoff: 2026-01\n"
    "Current date: 2026-01-22\n\n"
    "Reasoning: medium\n\n"
    "# Valid channels: analysis, final. Channel must be included for every message."
)

# --- 0. REPRODUCIBILITY ---
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

# --- 1. FILTERING ---
dataset = df.filter(
    pl.col("tools").is_null()
).drop(
    ["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url", "tools"], 
    strict=False
)

# Integer Filter
if KEEP_INT_ONLY:
    dataset = dataset.with_columns(
        pl.col("expected_answer")
        .str.extract(r"(-?\d+)", 1)
        .cast(pl.Int64, strict=False)
        .alias("numeric_value")
    ).filter(
        pl.col("numeric_value").is_not_null()
    )

# Optional uniqueness control
if cfg.KEEP_UNIQUE:
    unique_key = "problem" if "problem" in dataset.columns else "expected_answer"
    before_unique = len(dataset)
    dataset = dataset.unique(subset=[unique_key], keep="first")
    print(f"KEEP_UNIQUE=True | key={unique_key} | {before_unique} -> {len(dataset)}")

# Pre-sampling for speed
PRE_SAMPLE_SIZE = int(cfg.TRAIN_SIZE * 1.5)
if len(dataset) > PRE_SAMPLE_SIZE:
    dataset = dataset.sample(n=PRE_SAMPLE_SIZE, seed=cfg.SEED, shuffle=True)

# Enforce exact train size target
if len(dataset) > cfg.TRAIN_SIZE:
    dataset = dataset.sample(n=cfg.TRAIN_SIZE, seed=cfg.SEED, shuffle=True)
elif len(dataset) < cfg.TRAIN_SIZE:
    print(f"⚠️ Requested TRAIN_SIZE={cfg.TRAIN_SIZE}, available rows={len(dataset)}. Using available rows.")

# --- 2. CLEANUP ---
def clean_messages(messages):
    if messages is None:
        return []
    
    new_history = []
    
    for msg in messages:
        new_msg = dict(msg)
        
        # Skip Tool Roles
        if new_msg.get('role') == 'tool':
            continue
            
        # Clean artifacts
        new_msg.pop('tool_calls', None)
        new_msg.pop('tool_call_id', None)

        if new_msg.get('reasoning_content'):
            new_msg['thinking'] = new_msg.pop('reasoning_content')
            
        new_msg = {k: v for k, v in new_msg.items() if v is not None}
        new_history.append(new_msg)
        
    return new_history

print("Cleaning messages...")
dataset = dataset.with_columns(
    pl.col("messages").map_elements(clean_messages, return_dtype=pl.Object).alias("AA")
)

# --- 3. TOKENIZE (+ OPTIONAL SYSTEM PROMPT SWAP) ---
def replace_system_prompt_block(text, new_prompt):
    start_tag = "<|start|>system<|message|>"
    end_tag = "<|end|>"

    original_count = text.count(start_tag)
    if original_count == 0:
        return f"{start_tag}{new_prompt}{end_tag}" + text

    start_idx = text.find(start_tag)
    content_start = start_idx + len(start_tag)
    end_idx = text.find(end_tag, content_start)
    if end_idx == -1:
        raise ValueError("Malformed system block: missing <|end|> after system start tag.")

    updated = text[:content_start] + new_prompt + text[end_idx:]

    # Safety: do not change number of system tags; only replace first block content
    if updated.count(start_tag) != original_count:
        raise ValueError("System prompt replacement changed system tag count unexpectedly.")

    if new_prompt not in updated[content_start:content_start + len(new_prompt) + 4]:
        raise ValueError("System prompt replacement failed to place new prompt in first system block.")

    return updated

def apply_template_and_maybe_swap(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        reasoning_effort=cfg.REASONING_EFFORT
    )

    # Normal behavior path (same idea as low-training notebook): no prompt rewriting
    if not CHANGE_SYSTEM_PROMPT:
        return text

    # Optional custom system prompt path (robust first-system-block replacement)
    return replace_system_prompt_block(text, NEW_SYSTEM_PROMPT)

print(f"Tokenizing (System Prompt Swap: {CHANGE_SYSTEM_PROMPT})...")
dataset = dataset.with_columns(
    pl.col("AA").map_elements(apply_template_and_maybe_swap, return_dtype=pl.String).alias("text")
)

# --- 4. VERIFY ---
print("\n--- Final Prompt Check (First 500 chars) ---")
print(dataset["text"][0][:500])

if "developer" in dataset["text"][0][:500]:
    print("⚠️ WARNING: 'developer' role still found.")
else:
    print("✅ 'developer' role gone. System prompt is clean.")

print(f"Final dataset rows: {len(dataset)}")

Cleaning messages...
Tokenizing (System Prompt Swap: False)...

--- Final Prompt Check (First 500 chars) ---
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-19

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \boxed{}.

Given a concave function \( f(x) \), which is larger: \( (
✅ 'developer' role gone. System prompt is clean.
Final dataset rows: 10000


In [13]:
# Show answers that failed the strict number cast
dropped_rows = df.filter(
    pl.col("tools").is_null()
).filter(
    pl.col("expected_answer").cast(pl.Float64, strict=False).is_null()
).select(["expected_answer"]).head(20)

print("--- REJECTED ANSWERS (Sample) ---")
print(dropped_rows)

--- REJECTED ANSWERS (Sample) ---
shape: (20, 1)
┌─────────────────────────────────┐
│ expected_answer                 │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ \( 3^{13} - 3 \)                │
│ \;y\bigl(\sqrt{x^{2}+y^{2}}+x\… │
│ 30\;\text{by}\;27               │
│ \(\frac{2abc}{ab + bc + ca}\)   │
│ a = b                           │
│ …                               │
│ a=8                             │
│ \( a = 1.465, b = \frac{\pi}{6… │
│ \[                              │
│ x = \frac{S^2 + b^2 + c^2 -…    │
│ y=x+1                           │
│ $(-\infty ;4]\cup \left[\frac{… │
└─────────────────────────────────┘


In [14]:
import pyarrow as pa
from datasets import Dataset
hf_arrow_table = dataset.select(["text"]).to_arrow()

# 2. Create the Hugging Face Dataset directly from Arrow (Zero-Copy)
hf_dataset = Dataset(hf_arrow_table)


In [17]:
print(hf_dataset[20])

{'text': '<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-02-19\n\nReasoning: high\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: \'functions\'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nEvaluate the integral \\(\\int_{0}^{\\frac{\\pi }{2}}t^{2n}\\log^{m}\\left ( 2\\cos t  \\right )\\mathrm{d}t\\).<|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><|start|>assistant<|channel|>final<|message|><|end|><